# Session 6 lab — Non-relational databases and NoSQL

Three incident reports, stored twice: first in PostgreSQL tables, then as MongoDB
documents. Part C, optional, watches a PostgreSQL follower trail its leader.

To hand in: your part A schema with a note of what it cost, the two part B queries
with their SQL equivalents, and one or two sentences on which model you would
choose for this data.

In [ ]:
from datetime import datetime
from typing import LiteralString

import pandas as pd
import psycopg
from psycopg.sql import SQL, Identifier

PG = psycopg.connect("host=postgres user=labs password=labs dbname=labs")
PG.autocommit = True
# Your tables go in their own schema, away from the taxi tables.
PG.execute("SET search_path TO incidents")


def sql(statement: LiteralString) -> pd.DataFrame | None:
    """Run SQL on PostgreSQL, and show whatever rows come back."""
    if not statement.strip():
        print("nothing to run yet — write your SQL between the quotes")
        return None
    with PG.cursor() as cursor:
        cursor.execute(statement)
        if cursor.description is None:
            return None
        columns = [column.name for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)


def reset_incidents() -> None:
    """Start the schema empty, so the cell below can be run again."""
    PG.execute("DROP SCHEMA IF EXISTS incidents CASCADE")
    PG.execute("CREATE SCHEMA incidents")


def cost() -> pd.DataFrame:
    """For each of your tables: its columns, how many may be NULL, how many are."""
    tables = PG.execute(
        "SELECT table_name FROM information_schema.tables"
        " WHERE table_schema = 'incidents' ORDER BY table_name"
    ).fetchall()
    rows = []
    for (table,) in tables:
        nullable = PG.execute(
            "SELECT count(*) FILTER (WHERE is_nullable = 'YES'), count(*)"
            " FROM information_schema.columns"
            " WHERE table_schema = 'incidents' AND table_name = %s",
            (table,),
        ).fetchone() or (0, 0)
        everything = SQL("SELECT * FROM {}").format(Identifier("incidents", table))
        contents = pd.DataFrame(PG.execute(everything).fetchall())
        rows.append(
            {
                "table": table,
                "columns": nullable[1],
                "nullable columns": nullable[0],
                "rows": len(contents),
                "NULL cells": int(contents.isna().to_numpy().sum()),
            }
        )
    return pd.DataFrame(rows)

## Part A — The reports as a table (8 min)

Three reports from the taxi company's support desk:

> **INC-1001, a collision.** Reported on 14 January 2024 at 08:42, during trip
> 48213. The other vehicle's plate was KJM-4471, the police report number is
> 2024-0114-0331, and nobody was injured.
>
> **INC-1002, a lost item.** Reported on 17 January 2024 at 22:10, after trip
> 51877. The rider left a black umbrella, which the driver handed in at the
> Queens depot.
>
> **INC-1003, a fare dispute.** Reported on 21 January 2024 at 19:05, about trip
> 60342. The rider was charged $48.20 and was refunded $15.00.

The cell below stores them in one table. The shared facts and the collision's
already have columns. Add the columns the other two reports need, so that the
three `INSERT`s run.

In [ ]:
reset_incidents()
sql("""
CREATE TABLE incident (
    incident_id   text PRIMARY KEY,
    type          text NOT NULL,
    reported_at   timestamp NOT NULL,
    trip_id       bigint NOT NULL,
    other_plate   text,
    police_report text,
    injuries      boolean
    -- the lost item's and the fare dispute's columns
);

INSERT INTO incident (incident_id, type, reported_at, trip_id,
                      other_plate, police_report, injuries)
VALUES ('INC-1001', 'collision', '2024-01-14 08:42', 48213,
        'KJM-4471', '2024-0114-0331', false);
INSERT INTO incident (incident_id, type, reported_at, trip_id, item, depot)
VALUES ('INC-1002', 'lost item', '2024-01-17 22:10', 51877,
        'black umbrella', 'Queens');
INSERT INTO incident (incident_id, type, reported_at, trip_id, charged, refund)
VALUES ('INC-1003', 'fare dispute', '2024-01-21 19:05', 60342, 48.20, 15.00);
""")

In [ ]:
cost()

What did the design cost? A fourth type then arrives: a vehicle breakdown, with
the tow company and the repair cost. What changes?

*Nullable columns: … NULL cells: … A fourth type: …*

## Part B — The reports as documents (10 min)

From here on, MongoDB. A *collection* holds documents, much as a table holds rows,
but nothing fixes which fields a document has. In Python, a document is a
dictionary.

The collection is not empty: an older version of the support app left three
reports in it.

In [ ]:
from pymongo import MongoClient

MONGO = MongoClient("mongodb://mongodb:27017")
reports = MONGO["labs"]["incidents"]

# Written by the previous version of the support app.
OLDER_REPORTS = [
    {"_id": "INC-0907", "type": "complaint", "reported_at": datetime(2024, 1, 3, 14, 5),
     "trip_id": 1203, "about": "driver", "text": "On the phone for the whole ride."},
    {"_id": "INC-0912", "type": "complaint", "reported_at": datetime(2024, 1, 6, 9, 30),
     "trip_id": 2290, "about": "vehicle", "text": "The back seat was wet."},
    {"_id": "INC-0918", "type": "Complaint", "date": "2024-01-09", "trip": 3311,
     "text": "Refused to take a card."},
]


def reset_reports() -> None:
    """Empty the collection and put back the older reports."""
    reports.delete_many({})
    reports.insert_many(OLDER_REPORTS)


reset_reports()
print(reports.count_documents({}), "reports in the collection")

The same three reports, as documents. Each one carries only its own fields. Run
the cell.

In [ ]:
reset_reports()
reports.insert_many([
    {"_id": "INC-1001", "type": "collision", "reported_at": datetime(2024, 1, 14, 8, 42),
     "trip_id": 48213, "other_plate": "KJM-4471", "police_report": "2024-0114-0331",
     "injuries": False},
    {"_id": "INC-1002", "type": "lost item", "reported_at": datetime(2024, 1, 17, 22, 10),
     "trip_id": 51877, "item": "black umbrella", "depot": "Queens"},
    {"_id": "INC-1003", "type": "fare dispute", "reported_at": datetime(2024, 1, 21, 19, 5),
     "trip_id": 60342, "charged": 48.20, "refund": 15.00},
])
print(reports.count_documents({}), "reports in the collection")

Write each query twice: in MongoDB, then in SQL against your part A table.

**Query 1: the fare dispute, INC-1003.** `reports.find_one(filter)` returns the
first document matching a filter, written as a dictionary: `{"field": value}`.

In [ ]:
reports.find_one({})  # your filter between the braces

In [ ]:
sql("""

""")

**Query 2: the number of reports of each type.** `reports.distinct("type")` lists
the types; `reports.count_documents(filter)` counts the matches.

In [ ]:
counts: dict[str, int] = {}  # one entry per type
counts

In [ ]:
sql("""

""")

Was either model easier to query? Look at the counts: what happened to the older
reports, and would PostgreSQL have accepted them? Which model would you choose
for this data, and why?

*Your answer, in one or two sentences:*

## Part C — Replication lag (8 min, optional)

Two more PostgreSQL servers: `leader`, which takes the writes, and `follower`,
which copies them asynchronously. The leader confirms each write without waiting
for the follower.

The cell below writes a new value on the leader and reads it straight back from
the follower, 1,000 times, and counts how often the follower still had the old
value.

In [ ]:
LEADER = psycopg.connect("host=leader user=labs password=labs dbname=labs")
FOLLOWER = psycopg.connect("host=follower user=labs password=labs dbname=labs")
LEADER.autocommit = FOLLOWER.autocommit = True

LEADER.execute("DROP TABLE IF EXISTS counter")
LEADER.execute("CREATE TABLE counter (n integer)")
LEADER.execute("INSERT INTO counter VALUES (0)")
# Wait until the new table has reached the follower.
while FOLLOWER.execute("SELECT to_regclass('counter')").fetchone() == (None,):
    pass

TRIES = 1000
stale = 0
for n in range(1, TRIES + 1):
    LEADER.execute("UPDATE counter SET n = %s", (n,))
    seen = FOLLOWER.execute("SELECT n FROM counter").fetchone()
    if seen is None or seen[0] != n:
        stale += 1
print(f"{stale} of {TRIES} reads from the follower returned an old value")

LEADER.execute(
    "SELECT application_name, state, sync_state, replay_lag FROM pg_stat_replication"
).fetchall()

Run it two or three times. `replay_lag` shows the follower well under a
millisecond behind. Why are reads stale at all, and what would make the figure
worse: more writes at once, a long-running transaction on the leader, a slow
network between the two? What would a rider see if the app read their booking
back from the follower?

*Your answer:*